# TP Web Scraping Finance

**Question :** Le cours des actions du CAC40 change-t-il beaucoup le lendemain d'une grande annonce économique ?

In [23]:
!pip install selenium webdriver-manager beautifulsoup4 requests pandas


In [24]:
import time
import random
import requests
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager


##Les echos avec Selenium

In [25]:
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
driver.get("https://www.lesechos.fr/")

#obligation d'accepter les ccokie, sinon on ne peut pas cliquer sur l'onglet monde
cookies = WebDriverWait(driver, 8).until(
    EC.element_to_be_clickable((By.XPATH, "//button[text()='Accepter']"))
)
time.sleep(round(random.uniform(3, 5), 1))
cookies.click()

#on peut cliquer sur l'onglet monde
world_tab = driver.find_element(By.CSS_SELECTOR, "a[href='/monde']")
time.sleep(round(random.uniform(3, 5), 1))
world_tab.click()
time.sleep(round(random.uniform(3, 6), 1))


### Récupération des articles sur 3 mois (aujourd'hui → J-90)

In [26]:
import re
from datetime import date

# Période : aujourd'hui jusqu'à 3 mois en arrière
date_end  = date.today()
date_from = date_end - timedelta(days=90)

print(f"On récupère les articles du {date_from} au {date_end}")

# Dictionnaire mois français → numéro
mois_fr = {
    'janv': 1, 'févr': 2, 'mars': 3, 'avr': 4,
    'mai': 5, 'juin': 6, 'juil': 7, 'août': 8,
    'sept': 9, 'oct': 10, 'nov': 11, 'déc': 12
}

def parse_date_scraping(date_texte):
    # "Publié le 28 avr. à 19:50"
    match = re.search(r'(\d{1,2})\s+(\w+)\.?\s+à', date_texte)
    if match:
        jour, mois_str = match.groups()
        mois_num = mois_fr.get(mois_str.lower())
        if mois_num:
            return date(date.today().year, mois_num, int(jour))
    # "Publié à 15:40" → aujourd'hui
    if 'Publié à' in date_texte:
        return date.today()
    return None

articles = []
titres_deja_vus = set()
stop = False  # flag pour sortir de la boucle de pages quand on dépasse date_from

for page in range(1, 25):  # 25 pages max pour couvrir 3 mois
    if stop:
        break

    url = f"https://www.lesechos.fr/monde?page={page}"
    driver.get(url)
    time.sleep(round(random.uniform(3, 6), 1))
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    spans_articles = soup.find_all('a', class_='sc-luyfd4-0')

    if not spans_articles:
        print(f"Page {page} : plus d'articles, on s'arrête.")
        break

    print(f"Page {page} : {len(spans_articles)} balises <a> trouvées")

    for article in spans_articles:

        # On ne traite que les <a> qui contiennent un <h3> (lien texte, pas image)
        if not article.find('h3'):
            continue

        # Titre
        title = article.get('aria-label', '').strip()
        if not title or title in titres_deja_vus:
            continue

        # Date
        date_texte = ''
        for span in article.find_all('span'):
            texte = span.get_text(strip=True)
            if 'Publié' in texte:
                date_texte = texte
                break

        article_date = parse_date_scraping(date_texte)

        # Si l'article est plus vieux que 3 mois, on arrête tout
        if article_date and article_date < date_from:
            print(f"  → Article du {article_date} dépassé, on arrête.")
            stop = True
            break

        # Si l'article est dans la période, on l'ajoute
        if article_date and date_from <= article_date <= date_end:
            titres_deja_vus.add(title)
            articles.append({
                'titre':      title,
                'date_texte': date_texte,
                'date':       article_date,
            })

print(f"\nTotal : {len(articles)} articles (sans doublons) entre {date_from} et {date_end}")

On récupère les articles du 2026-02-03 au 2026-05-04
Page 1 : 180 balises <a> trouvées
Page 2 : 180 balises <a> trouvées
Page 3 : 181 balises <a> trouvées
Page 4 : 181 balises <a> trouvées
Page 5 : 181 balises <a> trouvées
Page 6 : 181 balises <a> trouvées
Page 7 : 181 balises <a> trouvées
Page 8 : 181 balises <a> trouvées
Page 9 : 181 balises <a> trouvées
Page 10 : 181 balises <a> trouvées
Page 11 : 181 balises <a> trouvées
Page 12 : 181 balises <a> trouvées
Page 13 : 181 balises <a> trouvées
Page 14 : 181 balises <a> trouvées
Page 15 : 181 balises <a> trouvées
Page 16 : 181 balises <a> trouvées
Page 17 : 181 balises <a> trouvées
Page 18 : 181 balises <a> trouvées
Page 19 : 180 balises <a> trouvées
  → Article du 2026-02-02 dépassé, on arrête.

Total : 877 articles (sans doublons) entre 2026-02-03 et 2026-05-04


In [27]:
df = pd.DataFrame(articles, columns=['titre', 'date_texte', 'date'])
df

,titre,date_texte,date
0,"Rudy Giuliani, l'ex-maire de New York et avoca...",Publié à 08:33,2026-05-04
1,DIRECT - Guerre au Moyen-Orient : l'armée amér...,Publié à 07:55,2026-05-04
2,Guerre en Ukraine : les Européens en force à E...,Publié à 06:10,2026-05-04
3,Guerre au Moyen-Orient : les Etats-Unis et l'I...,Publié le 3 mai à 18:20,2026-05-03
4,"Le Canada, invité inattendu au Sommet de la Co...",Publié le 3 mai à 17:56,2026-05-03
...,...,...,...
872,L'immigration légale aux Etats-Unis en chute l...,Publié le 3 févr. à 12:08,2026-02-03
873,"Gaz de combat, bases secrètes, arrestations ma...",Publié le 3 févr. à 12:07,2026-02-03
874,Droits de douane : les zones d'ombre que cache...,Publié le 3 févr. à 10:50,2026-02-03
875,L'Iran se prépare à des pourparlers sur le nuc...,Publié le 3 févr. à 07:58,2026-02-03


In [28]:
output_file = "articles.csv"
df.to_csv(output_file, index=False)
print(f"Fichier exporte : {output_file}")

Fichier exporte : articles.csv


In [29]:
# On ferme le navigateur
driver.quit()
print("Navigateur fermé.")

Navigateur fermé.
